In [1]:
from itertools import tee
from typing import Iterable, Optional, Union

import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from nlp.safe_input import safe_input
from nlp.config_nlm import INPUT_PRESETS, InputPreset
from nlp.sentence_sentiment_analyzer import SentimentViewer

`from itertools import tee`

**Purpose:**  
Import a tool that can duplicate an iterator.

**Behavior:**
- Allows one iterable to be split into two independent copies.
- Each copy can be consumed separately without losing data.

**Effect:**  
Makes it possible to “peek” at a generator without destroying it.

**Plain meaning:**  
Let the same stream of data be used twice.

---

`from typing import Iterable, Optional, Union`

**Purpose:**  
Provide type-hinting tools for clearer function signatures.

**Behavior:**
- `Iterable` describes “something you can loop over”.
- `Optional[T]` means “either `T` or `None`”.
- `Union[A, B]` means “either type `A` or type `B`”.

**Effect:**  
Improves readability and static analysis without changing runtime behavior.

**Plain meaning:**  
Describe what kinds of values functions expect and return.

---

`import pandas as pd`

**Purpose:**  
Bring in the Pandas library for working with CSV files and tables.

**Behavior:**
- Provides `read_csv`, `DataFrame`, and data-cleaning utilities.

**Effect:**  
Enables efficient loading and saving of structured text data.

**Plain meaning:**  
Handle CSV files and table-like data.  

---

`from nltk.sentiment.vader import SentimentIntensityAnalyzer`

**Purpose:**  
Import the VADER sentiment analyzer.

**Behavior:**
- Provides a ready-made model that scores text as positive, neutral, or negative.
- Returns a “compound” score between -1 and 1.

**Effect:**  
Enables automatic sentiment analysis of sentences.

**Plain meaning:**  
Measure how positive or negative a sentence is.  

---

`from nlp.safe_input import safe_input`

**Purpose:**  
Import a custom input helper function.

**Behavior:**
- Replaces raw `input()` with a safer, type-aware alternative.

**Effect:**  
Prevents invalid user input from crashing the program.

**Plain meaning:**  
Ask the user questions safely.

---

`from nlp.config_nlm import INPUT_PRESETS, InputPreset`

**Purpose:**  
Import predefined configuration data and its type.

**Behavior:**
- `INPUT_PRESETS` holds named or numbered default settings.
- `InputPreset` defines the structure of those settings.

**Effect:**  
Centralizes input defaults and makes them easy to switch.

**Plain meaning:**  
Load ready-made “settings bundles” the program can use.

---

`from nlp.sentence_sentiment_analyzer import SentimentViewer`

**Purpose:**  
Import the class called SentimentViewer for visualization.  

**Behavior:**  
- Runs the tkinter colour representation of the data.

**Effect:**  
Opens an interface that shows the given sentences in colour depedant of how positive or negative they are.  

**Plain meaning:**  
Loads the visualizer part of the code.


In [2]:
def load_csv_sentences(
    file_name: str = "comments.csv",
    col_name: str = "self_text",
    chunksize: int = 50000,
    limit: int = 0,
    multiline_bucket: Optional[list[str]] = None,
    treat_blankline_as_paragraph: bool = False,
) -> Optional[Iterable[str]]:
    if multiline_bucket is None:
        multiline_bucket = []

    try:
        _ = next(
            pd.read_csv(
                file_name,
                usecols=[col_name],
                encoding="utf-8",
                encoding_errors="ignore",
                chunksize=chunksize,
            )
        )
    except Exception:
        return None

    def is_multiparagraph(text: str) -> bool:
        t = text.replace("\r\n", "\n").replace("\r", "\n")
        return ("\n\n" in t) if treat_blankline_as_paragraph else ("\n" in t)

    def generator() -> Iterable[str]:
        count = 0
        row_offset = 0
        try:
            for chunk in pd.read_csv(
                file_name,
                usecols=[col_name],
                encoding="utf-8",
                encoding_errors="ignore",
                chunksize=chunksize,
            ):
                series = chunk[col_name].dropna()

                for local_i, s in series.items():
                    s = str(s).strip()
                    if not s:
                        continue

                    global_row = row_offset + int(local_i)
                    prefixed = f"{global_row} {s}"

                    if is_multiparagraph(s):
                        prefixed_fixed = prefixed.replace("\n", " ").replace("\r", " ")
                        multiline_bucket.append(prefixed)
                        yield prefixed_fixed
                        continue

                    yield prefixed
                    count += 1

                    if limit > 0 and count >= limit:
                        return

                row_offset += len(chunk)

            if count == 0:
                return
        except Exception:
            return

    g = generator()
    g_check, g_live = tee(g, 2)

    try:
        next(g_check)
    except StopIteration:
        return None

    return g_live


## `load_csv_sentences(file_name: str = "comments.csv", col_name: str = "self_text", chunksize: int = 50000, limit: int = 0, multiline_bucket: Optional[list[str]] = None, treat_blankline_as_paragraph: bool = False) -> Optional[Iterable[str]]`

**Purpose:**  
Load text rows from a CSV column as a stream (generator) of sentences.

**Behavior:**
- If `multiline_bucket` is not provided, creates an empty list for it.
- Tries reading the CSV once to confirm the file/column is readable; returns `None` if not.
- Detects “multi-paragraph” text based on newline rules.
- Reads the CSV in chunks to handle large files efficiently.
- Strips empty rows and skips blank text.
- Prefixes each sentence with its row number.
- For multi-paragraph entries, replaces newlines with spaces for the yielded version and stores the original in `multiline_bucket`.
- Stops early if `limit` is greater than 0 and enough rows were yielded.
- Uses `tee` to “peek” at the generator and returns `None` if it would produce nothing.

**Effect:**  
Returns either a usable iterable of sentences or `None` if the CSV cannot produce any.

**Plain meaning:**  
Try to pull sentences out of a CSV file without loading the whole file into memory.


In [3]:
def get_preset(debug_mode: int) -> InputPreset:
    preset = INPUT_PRESETS.get(debug_mode)
    if preset is None:
        raise ValueError(f"Invalid DEBUG_MODE: {debug_mode}")
    return preset

## `get_preset(debug_mode: int) -> InputPreset`

**Purpose:**  
Select a predefined set of input defaults based on a mode number.

**Behavior:**
- Looks up `debug_mode` in `INPUT_PRESETS`.
- Raises an error if the mode is not found.

**Effect:**  
Centralizes “default input choices” so the program can switch configurations easily.

**Plain meaning:**  
Pick a set of default settings based on a simple number.


In [4]:
def prompt_csv_inputs(preset: InputPreset) -> tuple[str, str, int]:
    file_name = safe_input(
        str,
        f"Enter CSV file name (default: {preset.file_default}): ",
        default=preset.file_default,
    )
    col_name = safe_input(
        str,
        f"Enter column name (default: {preset.col_default}): ",
        default=preset.col_default,
    )
    limit = safe_input(
        int,
        f"Enter max number of sentences to load (0 = all, default: {preset.limit_default}): ",
        default=preset.limit_default,
    )
    return file_name, col_name, limit

## `prompt_csv_inputs(preset: InputPreset) -> tuple[str, str, int]`

**Purpose:**  
Ask the user for CSV loading settings (file, column, limit).

**Behavior:**
- Prompts for file name, using the preset default if the user provides nothing or invalid input.
- Prompts for column name, using the preset default.
- Prompts for a sentence limit, using the preset default.

**Effect:**  
Returns the user’s choices in a consistent format.

**Plain meaning:**  
Ask the user “which file, which column, how many rows?”

In [5]:
def prompt_manual_sentences() -> list[str]:
    raw = safe_input(
        str,
        "No comments found. Enter sentences separated by ';': ",
        default="Hello World!;Subject is BAD BAD BAD!!;Subject is not too bad.;Subject is the best thing ever!",
    )
    return [s for s in raw.split(";") if s.strip()]

## `prompt_manual_sentences() -> list[str]`

**Purpose:**  
Get sentences from the user manually if CSV loading fails.

**Behavior:**
- Prompts for a single string where sentences are separated by `;`.
- Uses a built-in example string if the user provides nothing or invalid input.
- Splits the string on `;` and removes empty entries.

**Effect:**  
Produces a list of sentences even when there is no CSV data available.

**Plain meaning:**  
If there is no file data, just let the user type a few sentences.

In [6]:
def get_sentences(preset: InputPreset) -> Union[Iterable[str], list[str]]:
    try:
        file_name, col_name, limit = prompt_csv_inputs(preset)
        sentences = load_csv_sentences(file_name, col_name, limit=limit)
        if sentences is None:
            return prompt_manual_sentences()
        return sentences
    except Exception as e:
        print(f"Error loading CSV sentences: {e}")
        return prompt_manual_sentences()

## `get_sentences(preset: InputPreset) -> Union[Iterable[str], list[str]]`

**Purpose:**  
Obtain sentences either from a CSV file or from manual input.

**Behavior:**
- Prompts for CSV inputs.
- Attempts to load from CSV.
- If CSV loading returns `None`, falls back to manual entry.
- If any unexpected error happens, prints the error and falls back to manual entry.

**Effect:**  
Guarantees that the program ends up with something to analyze.

**Plain meaning:**  
Try to get sentences from a file; if that fails, ask the user instead.

In [7]:
def compute_average_compound(sentences: Iterable[str]) -> tuple[float, int]:
    sid = SentimentIntensityAnalyzer()
    total = 0.0
    count = 0
    for s in sentences:
        total += sid.polarity_scores(s)["compound"]
        count += 1
    avg = total / count if count > 0 else 0.0
    return avg, count

## `compute_average_compound(sentences: Iterable[str]) -> tuple[float, int]`

**Purpose:**  
Compute the average VADER compound sentiment score over many sentences.

**Behavior:**
- Creates a new sentiment analyzer.
- Loops over sentences, adding each compound score to a running total.
- Counts how many sentences were processed.
- Returns `(average, count)`.

**Effect:**  
Produces a single “overall sentiment” number plus how many items it was based on.

**Plain meaning:**  
Calculate the overall average sentiment across all sentences.


In [8]:
def average_line_text(avg: float, count: int) -> str:
    return f"Average compound sentiment over {count} sentences: {avg}"

## `average_line_text(avg: float, count: int) -> str`

**Purpose:**  
Create a readable sentence summarizing the average sentiment.

**Behavior:**
- Builds a formatted string including the count and average value.

**Effect:**  
Standardizes the summary message.

**Plain meaning:**  
Turn the average score into a human-readable summary line.

In [9]:
def to_csv(output_file: str, sentences: pd.DataFrame) -> None:
    try:
        sentences.to_csv(output_file, index=False, encoding="utf-8", sep=",")
        print(f"Sentences saved to {output_file}")
    except Exception as e:
        print(f"Error saving sentences to CSV: {e}")


## `to_csv(output_file: str, sentences: pd.DataFrame) -> None`

**Purpose:**  
Save a DataFrame of sentences to a CSV file.

**Behavior:**
- Writes the DataFrame to `output_file` as UTF-8 CSV without row indexes.
- Prints a success message if it works.
- Prints an error message if saving fails.

**Effect:**  
Creates an output CSV file containing the processed text.

**Plain meaning:**  
Write the results into a CSV file you can open later.

In [10]:
def run_pipeline(
    viewer: SentimentViewer,
    sentences: Union[Iterable[str], list[str]],
    *,
    average_only: bool,
) -> None:
    sentences = list(sentences)
    if average_only:
        avg, count = compute_average_compound(sentences)
        viewer.add_line(average_line_text(avg, count), score=avg)
        print(average_line_text(avg, count))
        viewer.run()
        return

    sid = SentimentIntensityAnalyzer()
    total = 0.0
    count = 0
    for s in sentences:
        ss = sid.polarity_scores(s)
        total += ss["compound"]
        count += 1
        viewer.add_line(s)

    avg = total / count if count > 0 else 0.0
    alt = average_line_text(avg, count)
    viewer.add_line(alt, score=avg)
    print(alt)

    viewer.analyzed_sentences.append(alt)
    v_sentences = viewer.analyzed_sentences

    df = pd.DataFrame(v_sentences)
    to_csv("sentiment_output.csv", df)

    viewer.run()

## `run_pipeline(viewer: SentimentViewer, sentences: Union[Iterable[str], list[str]], *, average_only: bool) -> None`

**Purpose:**  
Run the sentiment analysis workflow and display/output results.

**Behavior:**
- Forces `sentences` into a list so it can be reused multiple times.
- If `average_only` is true:
  - Computes the average score.
  - Adds a single colored “average” line to the viewer.
  - Prints the same summary to the console.
  - Starts the viewer and stops.
- Otherwise:
  - Analyzes each sentence and adds it to the viewer.
  - Computes and displays the final average line.
  - Builds a DataFrame from all displayed lines.
  - Saves them to `sentiment_output.csv`.
  - Starts the viewer.

**Effect:**  
Performs the full analysis and ensures results are visible and saved.

**Plain meaning:**  
Do the work: analyze text, show it, compute the overall average, and save the output.


In [11]:
def main(debug_mode: int = 1) -> None:
    preset = get_preset(debug_mode)

    sentences = get_sentences(preset)

    to_console = safe_input(
        bool, "Output to console? Y/n (default: False): ", default=False
    )
    average_only = safe_input(
        bool, "Compute average only? Y/n (default: False): ", default=False
    )
    viewer = SentimentViewer(title="Comment Sentiment Log", console_output=to_console)

    try:
        run_pipeline(viewer, sentences, average_only=average_only)
    except Exception as e:
        print(f"Error processing sentences: {e}")

## `main(debug_mode: int = 1) -> None`

**Purpose:**  
Tie everything together: settings, inputs, viewer setup, and running the pipeline.

**Behavior:**
- Gets a preset based on `debug_mode`.
- Gets sentences (CSV or manual).
- Asks whether to print results to the console.
- Asks whether to compute only the average.
- Creates the `SentimentViewer`.
- Runs the pipeline and prints an error if something goes wrong.

**Effect:**  
Provides a single entry point to run the program.

**Plain meaning:**  
Start the program, ask a few setup questions, then run sentiment analysis.

In [12]:
main(1)

Enter CSV file name (default: Yasmins_comments.csv):  
Enter column name (default: Content):  
Enter max number of sentences to load (0 = all, default: 100):  
Output to console? Y/n (default: False):  
Compute average only? Y/n (default: False):  


Average compound sentiment over 106 sentences: 0.4356792452830186
Sentences saved to sentiment_output.csv
